# Translation Task

[video](https://www.youtube.com/watch?v=ISNdQcPhsts)

In [1]:
from pathlib import Path

from dotenv import load_dotenv
from torch.utils.tensorboard import SummaryWriter

import models.deep_learning.architectures.transformer.tasks.translation as trn

load_dotenv()
device = trn.get_device()

## Config

In [2]:
CONFIG = trn.Config(
    batch_size=8,
    num_epochs=50,
    lr=1e-4,
    src_seq_len=350,
    tgt_seq_len=350,
    d_model=512,
    datasource="Helsinki-NLP/opus_books",
    src_lang="en",
    tgt_lang="es",
    model_basename="tmodel_",
)


In [3]:
writer = SummaryWriter(CONFIG.experiment_name)
Path(CONFIG.model_folder).mkdir(parents=True, exist_ok=True)

## Load Dataset (from HuggingFace)

In [4]:
raw_ds = trn.TranslationHFDataset.load_dataset(
    path=CONFIG.datasource,
    name=f"{CONFIG.src_lang}-{CONFIG.tgt_lang}",
    split="train",
)

## Tokenization

In [5]:
tokenizer_src = trn.get_or_build_tokenizer(
    Path(CONFIG.tokenizer_src_file), raw_ds, CONFIG.src_lang
)
tokenizer_tgt = trn.get_or_build_tokenizer(
    Path(CONFIG.tokenizer_tgt_file), raw_ds, CONFIG.tgt_lang
)

## Create dataloaders

In [6]:
train_dataloader, val_dataloader = trn.create_dataloaders(
    raw_ds, tokenizer_src, tokenizer_tgt, CONFIG
)

Filter:   0%|          | 0/93470 [00:00<?, ? examples/s]

Original dataset size: 93470
Filtered dataset size: 93464
Removed: 6 samples (0.01%)


## Create model

In [7]:
model = trn.Translator(
    src_vocab_size=tokenizer_src.get_vocab_size(),
    tgt_vocab_size=tokenizer_tgt.get_vocab_size(),
    src_max_length=CONFIG.src_seq_len,
    tgt_max_length=CONFIG.tgt_seq_len,
    embed_size=CONFIG.d_model,
).to(device)

## Train the model

In [ ]:
trn.train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    tokenizer_src=tokenizer_src,
    tokenizer_tgt=tokenizer_tgt,
    device=device,
    config=CONFIG,
    writer=writer,
)

Preloading model Helsinki-NLP/opus_books_weights_en_es/tmodel_01.pt


Processing Epoch 02:   0%|          | 43/10515 [00:30<1:58:45,  1.47it/s, loss=4.555]